####  Reference Documentation

In [126]:
# https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/ai-services/language-service/custom-named-entity-recognition/how-to/call-api.md#tab/client
# https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/textanalytics/azure-ai-textanalytics/samples/sample_recognize_custom_entities.py
# https://learn.microsoft.com/en-us/python/api/overview/azure/ai-textanalytics-readme?view=azure-python
# https://learn.microsoft.com/en-us/azure/ai-services/language-service/summarization/quickstart?tabs=text-summarization%2Cwindows&pivots=programming-language-python

####  Loading Libraries and clients

In [127]:
import os
import json
import time
import openai
import requests
import pandas as pd
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, ContentFormat
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, DocumentAnalysisFeature

from azure.ai.textanalytics import (
    TextAnalyticsClient,
    ExtractiveSummaryAction,
    AbstractiveSummaryAction,
) 

from dotenv import load_dotenv
from openai import AzureOpenAI
load_dotenv(override=True)

aoai_client = AzureOpenAI(
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"), 
  api_key=os.getenv("AZURE_OPENAI_API_KEY"),  
  api_version="2024-07-01-preview"
)

AZURE_LANGUAGE_ENDPOINT = os.environ["AZURE_LANGUAGE_ENDPOINT"]
AZURE_LANGUAGE_KEY = os.environ["AZURE_LANGUAGE_KEY"]
CUSTOM_ENTITIES_PROJECT_NAME = os.environ["CUSTOM_ENTITIES_PROJECT_NAME"]
CUSTOM_ENTITIES_DEPLOYMENT_NAME = os.environ["CUSTOM_ENTITIES_DEPLOYMENT_NAME"]

# Text Analytics Client
text_analytics_client = TextAnalyticsClient(
    endpoint=AZURE_LANGUAGE_ENDPOINT,
    credential=AzureKeyCredential(AZURE_LANGUAGE_KEY),
)

# Document Intelligence Client
AZURE_DOC_INTELLIGENCE_ENDPOINT = os.environ["AZURE_DOC_INTELLIGENCE_ENDPOINT"]
AZURE_DOC_INTELLIGENCE_KEY = os.environ["AZURE_DOC_INTELLIGENCE_KEY"]
document_intelligence_client = DocumentIntelligenceClient(endpoint=AZURE_DOC_INTELLIGENCE_ENDPOINT, credential=AzureKeyCredential(AZURE_DOC_INTELLIGENCE_KEY), api_version="2024-02-29-preview")

# Azure OpenAI Client
aoai_client = AzureOpenAI(
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"), 
  api_key=os.getenv("AZURE_OPENAI_API_KEY"),  
  api_version="2024-07-01-preview"
)

#### Document Intelligence OCR

In [128]:
def text_html_processing(OcrExtractionDIOutput):
    offset = 0
    page_map = []
    page_map_dict =[]

    for page_num, page in enumerate(OcrExtractionDIOutput.pages):
        tables_on_page = [
            table
            for table in (OcrExtractionDIOutput.tables or [])
            if table.bounding_regions and table.bounding_regions[0].page_number == page_num + 1
        ]
        #print(tables_on_page)

        # mark all positions of the table spans in the page
        page_offset = page.spans[0].offset
        page_length = page.spans[0].length
        table_chars = [-1] * page_length
        for table_id, table in enumerate(tables_on_page):
            for span in table.spans:
                # replace all table spans with "table_id" in table_chars array
                for i in range(span.length):
                    idx = span.offset - page_offset + i
                    if idx >= 0 and idx < page_length:
                        table_chars[idx] = table_id

        # build page text by replacing characters in table spans with table html
        page_text = ""
        added_tables = set()
        for idx, table_id in enumerate(table_chars):
            if table_id == -1:
                page_text += OcrExtractionDIOutput.content[page_offset + idx]
            elif table_id not in added_tables:
                page_text += table_to_html(tables_on_page[table_id])
                added_tables.add(table_id)

        page_text += " "
        page_map.append((page_num+1, offset, page_text))

        single_page_dict = {}
        single_page_dict['page_num']= page_num+1
        single_page_dict['content'] = page_text
        single_page_dict['offset'] = offset
        page_map_dict.append(single_page_dict)

        offset += len(page_text)

    return page_map_dict

In [129]:
def OcrExtractionDI(relative_path: str, Markdown: [bool]=True):
    
    path_to_document = os.path.abspath(
        os.path.join(relative_path))
    
    if Markdown==True:
        output_format = ContentFormat.MARKDOWN
    else:
        output_format = None

    with open(path_to_document, "rb") as f:
        poller = document_intelligence_client.begin_analyze_document("prebuilt-layout", 
                                                                    analyze_request=f, content_type="application/octet-stream", 
                                                                    output_content_format=output_format)
    OcrExtractionDIOutput = poller.result()
    
    if Markdown==False:
        pagemap = text_html_processing(OcrExtractionDIOutput)
        extracted_processed_text = pagemap
    else:
        extracted_processed_text = OcrExtractionDIOutput

    return extracted_processed_text

In [130]:
def MdFormatting(ocr_extraction):
    doc_string = ocr_extraction.content
    strings_to_replace = re.findall(".+\n===", doc_string)
    for string in strings_to_replace:
        doc_string = doc_string.replace(string, "=== "+string.replace("===",""))

    ## Split the document into chunks base on markdown headers.
    headers_to_split_on = [
        ("===", "Title"),
        ("##", "Header 1"),
    ]
    text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

    markdown_chunks = text_splitter.split_text(doc_string)

    chunk_list = []
    for chunk in markdown_chunks:
        try:
            title = chunk.metadata['Title']
        except:
            title = ""
        try:
            header1 = chunk.metadata['Header 1']
        except:
            header1 = ""

        chunk_list.append({"title": title,"header":header1,"content": title + "/n" + header1 + "/n"+ chunk.page_content})
    return pd.DataFrame(chunk_list)

In [131]:
relative_path = "data/raw/"
Filename = "10Q-MSFT-04-25-2023.pdf"

Markdown = True

ocr_extraction = OcrExtractionDI(relative_path = relative_path+Filename, Markdown=Markdown)
markdown_extraction = ocr_extraction.content

In [132]:
len(ocr_extraction.content)

231028

#### Using Summarization API

In [137]:
def GetSummaries(document):
    
    headers = {"Ocp-Apim-Subscription-Key": AZURE_LANGUAGE_KEY, "Content-Type": "application/json"}

    url = f"{AZURE_LANGUAGE_ENDPOINT}/language/analyze-text/jobs?api-version=2023-11-15-preview"
    body = {
    "displayName": "Summarization",
    "analysisInput": {
        "documents": [
            {
                "id": "1",
                "language": "en",
                "text": document
            }]
    },
    "tasks": [
        {
        "kind": "AbstractiveSummarization",
        "taskName": "Text Abstractive Summarization Task 1",
        "parameters":{
            "summaryLength": "long"}
        },
        {
        "kind": "ExtractiveSummarization",
        "taskName": "Text Extractive Summarization Task 2",
        "parameters":{
            "sentenceCount": "20"}
        }
        ]
    }
    response = requests.post(url, headers=headers, json=body)

    try:
        get_url = response.headers.pop("operation-location")
    except:
        pass
    get_response = requests.get(get_url, headers=headers)
    status = get_response.json()['status']

    while status != "succeeded":
        time.sleep(2)
        get_response = requests.get(get_url, headers=headers)
        status = get_response.json()['status']
    
    get_summaries = get_response.json()

    return get_summaries

In [138]:
response = GetSummaries(markdown_extraction[:100000])

In [140]:
response

{'jobId': 'ec155a8f-897f-49ad-a501-f11ccde0323b',
 'lastUpdatedDateTime': '2024-10-21T17:32:41Z',
 'createdDateTime': '2024-10-21T17:32:25Z',
 'expirationDateTime': '2024-10-22T17:32:25Z',
 'status': 'succeeded',
 'errors': [],
 'displayName': 'Summarization',
 'tasks': {'completed': 2,
  'failed': 0,
  'inProgress': 0,
  'total': 2,
  'items': [{'kind': 'AbstractiveSummarizationLROResults',
    'taskName': 'Text Abstractive Summarization Task 1',
    'lastUpdateDateTime': '2024-10-21T17:32:30.6141855Z',
    'status': 'succeeded',
    'results': {'documents': [{'summaries': [{'text': "The document is a filing by Microsoft Corporation to the United States Securities and Exchange Commission, providing an overview of its financial performance for the period ending March 31, 2023. The company's unaudited interim consolidated financial statements, prepared in accordance with US Generally Accepted Accounting Principles (GAAP), show an increase in the estimated useful lives of server and netw

In [123]:
print(response['tasks']['items'][0]['results']['documents'][0]['summaries'][0]['text'])

The United States Securities and Exchange Commission (SEC) document outlines the financial status and compliance of Microsoft Corporation, as of March 31, 2023. It confirms that Microsoft has filed all required reports under Section 13 or 15(d) of the Securities Exchange Act of 1934 during the preceding 12 months and has been subject to such filing requirements for the past 90 days. Microsoft is classified as a non-accelerated filer and is not a shell company. The document also details Microsoft's adherence to GAAP in preparing its financial statements, which include estimates and assumptions affecting reported assets, liabilities, revenue, and expenses. A significant change in the estimated useful lives of server and network equipment from four to six years led to an increase in operating income and net income by $885 million and $720 million, respectively. Derivative instruments are recognized at fair value, with certain assets and liabilities measured accordingly. Equity investments